# IDEA on Kaggle GPUs (open-weight models)

Runs the open-weight stages: vignette checks, perceived stance, main elicitation with activation capture, and RQ3/RQ3b steering + the benchmark audit.

**Setup (once)**
1. On your Mac: `scripts/make_kaggle_bundle.sh` -> upload `dist/idea-code.zip` as a Kaggle Dataset named **idea-code** (it contains the code, the data files and the pre-generated `trials.parquet`, so every machine uses identical trials).
2. Notebook settings: Accelerator **GPU T4 x2**, Internet **on**, Persistence **Files only** (or use the resume input below).
3. Add-ons -> Secrets: `HF_TOKEN` (needed for the gated `meta-llama/Llama-3.1-8B-Instruct`; accept its license on Hugging Face first).

**Each session** (12 h limit): set `CONFIG`, `MODELS` and `STAGES` below, then *Save Version -> Save & Run All (Commit)*. Everything is cached by prompt hash, so a session that stops at the deadline resumes exactly where it left off: attach the previous version's output as an input and set `RESUME_FROM`.

Approximate GPU time (T4 x2, fp16, model split across both GPUs): ~5-7 h per full chat run of a 7-8B model (~135k forward passes); ~2.5-3.5 h per labels-only run.

In [ ]:
# ---- settings ----
CONFIG = "configs/pilot.yaml"          # then configs/main.yaml
MODELS = ["llama"]             # e.g. ["olmo_base", "olmo_sft"], ["olmo_dpo", "olmo_instruct"], ["llama"]
STAGES = ["cues", "stance", "elicit"]  # later: ["fit", "steer", "audit"] for olmo_instruct / llama
HOURS = 11.3                           # stop starting new chunks after this many hours (Kaggle kills at 12)
RESUME_FROM = None                     # e.g. "/kaggle/input/idea-olmo-run-v1/idea/runs"  (previous version's output)
API_RESULTS = None                     # e.g. "/kaggle/input/idea-api-results" (responses/stance/cues from GPT+Claude, for `fit`)
BATCH_SIZE = 16

In [ ]:
import os, shutil, subprocess, time, glob, sys
os.environ["IDEA_DEADLINE"] = str(time.time() + HOURS * 3600)
hits = sorted(glob.glob("/kaggle/input/**/idea-code/pyproject.toml", recursive=True) + glob.glob("/kaggle/input/**/pyproject.toml", recursive=True))
assert hits, "idea-code dataset not attached"
src = os.path.dirname(hits[0]); print("code from", src)
if not os.path.exists("/kaggle/working/idea"):
    shutil.copytree(src, "/kaggle/working/idea")
os.chdir("/kaggle/working/idea")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[hf,audit]"], check=True)
# a package installed in this session is not importable until restart; also /kaggle/working/idea would shadow it
sys.path.insert(0, "/kaggle/working/idea/src")
for k in [k for k in sys.modules if k == "idea" or k.startswith("idea.")]: del sys.modules[k]
import idea.pipeline; print("idea package from", idea.pipeline.__file__)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("no HF_TOKEN secret (needed for Llama):", e)
    if "llama" in MODELS:
        raise SystemExit("STOP: attach the HF_TOKEN secret (Add-ons -> Secrets) and run again")
if RESUME_FROM:
    shutil.copytree(RESUME_FROM, "runs", dirs_exist_ok=True)
    print("resumed from", RESUME_FROM)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import yaml, pathlib
cfg = yaml.safe_load(open(CONFIG))
cfg.setdefault("elicit", {})["batch_size"] = BATCH_SIZE
cfg["elicit"]["dtype"] = "float16"     # T4 has no bf16
pathlib.Path("configs/_kaggle.yaml").write_text(yaml.safe_dump(cfg))
from idea.pipeline import run_dir, load_config, config_hash
assert config_hash(load_config(CONFIG)) == config_hash(load_config("configs/_kaggle.yaml"))  # same run folder as the bundle
RUN = run_dir(load_config("configs/_kaggle.yaml"))
print("run folder:", RUN, "| trials present:", (RUN / "trials.parquet").exists())
if not (RUN / "trials.parquet").exists():
    !idea generate --config configs/_kaggle.yaml

In [ ]:
from idea.models import get_model
C = "--config configs/_kaggle.yaml"
for m in MODELS:
    spec = get_model(m)
    if "cues" in STAGES and "chat" in spec.formats:
        !idea cues {C} --model {m}
    if "stance" in STAGES:
        for fmt in spec.formats:
            !idea stance {C} --model {m} --fmt {fmt}
    if "elicit" in STAGES:
        for fmt in spec.formats:
            !idea elicit {C} --model {m} --fmt {fmt}

In [ ]:
# After all models (open + API) have responses: merge API results, then fit (CPU is fine).
if API_RESULTS:
    for sub in ("responses", "stance", "cues"):
        if os.path.exists(f"{API_RESULTS}/{sub}"):
            shutil.copytree(f"{API_RESULTS}/{sub}", RUN / sub, dirs_exist_ok=True)
if "screen" in STAGES:
    !idea screen {C}
if "pilot" in STAGES:
    !idea pilot {C}
if "fit" in STAGES:
    !idea fit {C}

In [ ]:
# RQ3 / RQ3b and the benchmark audit (open-weight final models only).
for m in MODELS:
    if "steer" in STAGES:
        !idea steer {C} --model {m}
    if "audit" in STAGES:
        !idea audit {C} --model {m}

In [ ]:
# Keep the output small enough for Kaggle (20 GB): drop activation shards only when you are done with RQ3.
!du -sh {RUN}/* 2>/dev/null | sort -h | tail